[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/03a_baseline.ipynb)

# 03a — Baselines: Random Search and a Rule-Based Sweep

**Question.** What does the tilt space offer without a model?

- **Random search** evaluates Sobol points over the tilt box, with the same evaluation budget and the same
  initial design as TuRBO. Any gap between the two is what the model contributes.
- **Rule-based sweep** gives every cell on a band the same tilt and tunes the three band tilts by coordinate
  descent, the way an operator would. It is deterministic and not budget-matched.

**Outputs.** One run directory per method and seed under `outputs/optim/`. **Requirements.** A CUDA GPU.
Random search runs once per seed in `SEEDS` (`optim.seed`, the global `seed` in `configs/config.yaml`, unless `BAND_TILT_SEEDS` is set); the shell equivalent is `task baseline`.

In [1]:
# Environment: locally, move to the project root; on Colab, clone the repository
# and install what Colab lacks. Extra Hydra overrides come from BAND_TILT_OVERRIDES.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna.rt", "sionna-rt")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    root = Path("/content/band-tilt")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    missing = [pip for module, pip in COLAB_PACKAGES if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
sys.path.insert(0, str(root))
CONFIG_OVERRIDES = os.environ.get("BAND_TILT_OVERRIDES", "").split()

In [2]:
%load_ext autoreload
%autoreload 2

from functools import partial

import pandas as pd

from src.config import load_config
from src.evaluation import compare
from src.evaluation import runs as run_store
from src.evaluation.export import readable, save_table
from src.optim.run import run
from src.optim.space import TiltSpace
from src.utils.plotting import label, save_fig, setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path("reports/figures/03a_baseline"))
save_table = partial(save_table, in_colab=IN_COLAB, directory=Path("reports/tables/03a_baseline"))
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 40)

# Search seeds: optim.seed from the config; BAND_TILT_SEEDS (space-separated) overrides it.
SEEDS = [int(seed) for seed in os.environ.get("BAND_TILT_SEEDS", str(cfg.optim.seed)).split()]

## 1. Search Space

In [3]:
space = TiltSpace.from_config(cfg)
search_space = (
    space.as_frame(space.baseline)
    .groupby("band", sort=False)
    .agg(
        cells=("cell", "size"),
        current=("tilt_deg", "median"),
        minimum=("tilt_min_deg", "min"),
        maximum=("tilt_max_deg", "max"),
    )
    .reset_index()
    .rename(
        columns={
            "cells": "Cells",
            "current": "Current tilt [°]",
            "minimum": "Minimum tilt [°]",
            "maximum": "Maximum tilt [°]",
        }
    )
)
search_space = readable(search_space)
save_table(search_space, "search_space")
print(f"{space.n_dim} decision variables: {len(space.cells)} cells x {len(space.band_names)} bands")
search_space

27 decision variables: 9 cells x 3 bands


,Band,Cells,Current tilt [°],Minimum tilt [°],Maximum tilt [°]
0,2600 MHz,9,10.0,0.0,15.0
1,1800 MHz,9,10.0,0.0,15.0
2,700 MHz,9,10.0,0.0,15.0


## 2. Runs

Each run evaluates the current configuration first, then searches, then archives its best configuration. A run already on disk for the same method and seed is reused; delete its directory under `outputs/optim/` to repeat it.

In [4]:
def on_disk():
    """The newest finished run of each method and seed."""
    return run_store.latest_per_method_and_seed(run_store.discover(cfg.optim.output.dir))


done = {(r.method, r.seed) for r in on_disk()}
for seed in SEEDS:
    if ("random", seed) not in done:
        run(load_config(overrides=["optim/method=random", f"optim.seed={seed}", *CONFIG_OVERRIDES]))

# Deterministic, so one run; its seed only labels it.
if not any(method == "rule" for method, _ in done):
    run(load_config(overrides=["optim/method=rule", f"optim.seed={SEEDS[0]}", *CONFIG_OVERRIDES]))

runs = [r for r in on_disk() if r.method == "rule" or (r.method == "random" and r.seed in SEEDS)]

jitc_llvm_init(): LLVM API initialization failed ..



random: 8 solutions published

 solution  recommended  score  hole_rate  overlap_rate  served_ratio  weak_rate  edge_rsrp_dbm  hole_desirability  overlap_desirability  served_desirability
        0        False 0.8499     0.1768        0.3212        0.9065     0.1739      -105.9248             0.8227                0.8886               0.8568
        1         True 0.8562     0.1755        0.3186        0.9115     0.1549      -104.6967             0.8249                0.9003               0.8659
        2        False 0.8556     0.1743        0.3325        0.9132     0.1491      -104.4364             0.8255                0.8952               0.8702
        3        False 0.8552     0.1748        0.3276        0.9148     0.1564      -105.3193             0.8249                0.8958               0.8685
        4        False 0.8552     0.1732        0.3325        0.9114     0.1606      -105.7436             0.8265                0.8961               0.8629
        5        False 0.8


rule: 8 solutions published

 solution  recommended  score  hole_rate  overlap_rate  served_ratio  weak_rate  edge_rsrp_dbm  hole_desirability  overlap_desirability  served_desirability
        0        False 0.8499     0.1768        0.3212        0.9065     0.1739      -105.9248             0.8227                0.8886               0.8568
        1         True 0.8574     0.1719        0.3314        0.9100     0.1422      -103.9616             0.8281                0.8977               0.8685
        2        False 0.8573     0.1728        0.3285        0.9125     0.1422      -103.8991             0.8273                0.8972               0.8711
        3        False 0.8571     0.1724        0.3345        0.9082     0.1458      -104.2541             0.8279                0.8975               0.8675
        4        False 0.8566     0.1737        0.3307        0.9107     0.1506      -104.5103             0.8263                0.8988               0.8666
        5        False 0.856

## 3. Results per Run

The weighted score (`configs/kpi.yaml` `weights`) selects each run's best configuration.

In [5]:
incumbent = runs[0].incumbent_kpi

baseline_results = readable(compare.method_table(runs, cfg).drop(columns=["run"]))
save_table(baseline_results, "baseline_results")
print("Current configuration:", {label(k): round(v, 4) for k, v in incumbent.as_dict().items()})
baseline_results

Current configuration: {'Coverage hole rate': 0.1768, 'Co-band overlap rate': 0.3212, 'Served UE ratio': 0.9065, 'Weak coverage rate': 0.1739, 'Cell-edge RSRP': -105.9248, 'Coverage desirability': 0.8227, 'Layer separation desirability': 0.8886, 'Served desirability, per tile': 0.8568}


,Method,Seed,Evaluations,Best evaluation,Ray tracing [min],Wall clock [min],Coverage hole rate,Co-band overlap rate,Served UE ratio,Weak coverage rate,Cell-edge RSRP,Coverage desirability,Layer separation desirability,"Served desirability, per tile",Quality index,KPIs improved,KPIs worsened
0,Random search,42,145,18,7.3080,8.8794,0.1755,0.3186,0.9115,0.1549,-104.6967,0.8249,0.9003,0.8659,0.8562,8,0
1,Rule-based sweep,42,28,12,1.1436,1.4525,0.1719,0.3314,0.9100,0.1422,-103.9616,0.8281,0.8977,0.8685,0.8574,7,1


**Observations.** _To be written._